# 2 - Generate Model Outputs

- Base model rows use completion/prefix prompts.
- Instruction-tuned rows use instruction prompts and are wrapped with the tokenizer chat template when available.

The input file is `generation_prompts_400.csv`, produced by the revised dataset notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from src.config import (
    BASE_MODEL,
    INSTRUCT_MODEL,
    MODEL_FAMILY,
    RAW_OUTPUTS_FILENAME,
    GENERATION_PROMPTS_FILENAME,
    get_data_dir,
)
from src.generation import (
    GenerationSettings,
    run_generation_pipeline,
    validate_generation_prompts,
    validate_raw_outputs,
)

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = get_data_dir(PROJECT_DIR)
GENERATION_PROMPTS_PATH = DATA_DIR / GENERATION_PROMPTS_FILENAME
RAW_OUTPUTS_PATH = DATA_DIR / RAW_OUTPUTS_FILENAME
print("Base model:", BASE_MODEL)
print("Instruction-tuned model:", INSTRUCT_MODEL)

Project directory: c:\Users\ayata\Desktop\p9_dataset_prompt_updates


## Load generation prompts

This file should contain 400 rows: 200 completion prompts for the base model and 200 instruction prompts for the instruction-tuned model.

In [11]:
generation_prompts_df = pd.read_csv(GENERATION_PROMPTS_PATH)
validate_generation_prompts(generation_prompts_df, expected_prompt_cases=200)

print("Loaded generation prompts:", generation_prompts_df.shape)
print(generation_prompts_df["model_type"].value_counts())
print(generation_prompts_df["input_format"].value_counts())
display(generation_prompts_df.head(3))

Loaded generation prompts: (400, 9)
model_type
base        200
instruct    200
Name: count, dtype: int64
input_format
completion     200
instruction    200
Name: count, dtype: int64


,generation_prompt_id,prompt_id,model_type,input_format,domain,topic,prompt_type,prompt_text,ideological_axis
0,economy_wealth_tax_neutral_arguments_base,economy_wealth_tax_neutral_arguments,base,completion,economy,wealth_tax,neutral_arguments,The debate about increasing taxes on wealthy i...,left_right
1,economy_wealth_tax_progressive_framed_base,economy_wealth_tax_progressive_framed,base,completion,economy,wealth_tax,progressive_framed,Many progressive policymakers argue that highe...,left_right
2,economy_wealth_tax_conservative_framed_base,economy_wealth_tax_conservative_framed,base,completion,economy,wealth_tax,conservative_framed,Many conservative policymakers argue that high...,left_right


## Check one base prompt and one instruction prompt

In [12]:
example_base = generation_prompts_df.loc[generation_prompts_df["model_type"].eq("base")].iloc[0]
example_instruct = generation_prompts_df.loc[generation_prompts_df["model_type"].eq("instruct")].iloc[0]

print("BASE COMPLETION PROMPT:")
print(example_base.prompt_text)

print("\nINSTRUCTION PROMPT:")
print(example_instruct.prompt_text)

BASE COMPLETION PROMPT:
The debate about increasing taxes on wealthy individuals has several arguments on both sides. A balanced explanation is that

INSTRUCTION PROMPT:
What are the main arguments for and against increasing taxes on wealthy individuals? Answer in a balanced way.


## Generation settings

Generation is deterministic: sampling is disabled, temperature is 0.0, and the random seed is fixed.

In [13]:
settings = GenerationSettings()
print(settings)

GenerationSettings(seed=42, max_new_tokens=128, do_sample=False, temperature=0.0)


## Run generation

In [14]:
raw_outputs = run_generation_pipeline(
    generation_prompts=generation_prompts_df,
    output_path=RAW_OUTPUTS_PATH,
    settings=settings,
    expected_prompt_cases=200,
)

print("Raw output shape:", raw_outputs.shape)
display(raw_outputs.head())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Generating with instruct: 100%|██████████| 200/200 [1:06:39<00:00, 20.00s/it]


Raw output shape: (400, 16)


,generation_prompt_id,prompt_id,domain,topic,prompt_type,input_format,ideological_axis,prompt_text,model_family,model_type,model_name,response_text,generation_seed,temperature,do_sample,max_new_tokens
0,economy_wealth_tax_neutral_arguments_base,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,completion,left_right,The debate about increasing taxes on wealthy i...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,the wealthy are not contributing enough to the...,42,0.0,False,128
1,economy_wealth_tax_progressive_framed_base,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,completion,left_right,Many progressive policymakers argue that highe...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is not a sound policy. The main reason is t...,42,0.0,False,128
2,economy_wealth_tax_conservative_framed_base,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,completion,left_right,Many conservative policymakers argue that high...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is not a sound policy. The tax on wealth is...,42,0.0,False,128
3,economy_wealth_tax_policy_recommendation_base,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,completion,left_right,A balanced explanation of the policy debate ar...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is necessary to protect the public from the...,42,0.0,False,128
4,economy_corporate_tax_neutral_arguments_base,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,completion,left_right,The debate about raising corporate tax rates h...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,the tax rate is a tax on the productive capaci...,42,0.0,False,128


## Validate saved outputs

In [15]:
raw_outputs = pd.read_csv(RAW_OUTPUTS_PATH)
validate_raw_outputs(raw_outputs, expected_prompt_cases=200)

print("Generated output validation passed.")
print("Shape:", raw_outputs.shape)
print(raw_outputs["model_type"].value_counts())
display(raw_outputs.head())

Generated output validation passed.
Shape: (400, 16)
model_type
base        200
instruct    200
Name: count, dtype: int64


,generation_prompt_id,prompt_id,domain,topic,prompt_type,input_format,ideological_axis,prompt_text,model_family,model_type,model_name,response_text,generation_seed,temperature,do_sample,max_new_tokens
0,economy_wealth_tax_neutral_arguments_base,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,completion,left_right,The debate about increasing taxes on wealthy i...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,the wealthy are not contributing enough to the...,42,0.0,False,128
1,economy_wealth_tax_progressive_framed_base,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,completion,left_right,Many progressive policymakers argue that highe...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is not a sound policy. The main reason is t...,42,0.0,False,128
2,economy_wealth_tax_conservative_framed_base,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,completion,left_right,Many conservative policymakers argue that high...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is not a sound policy. The tax on wealth is...,42,0.0,False,128
3,economy_wealth_tax_policy_recommendation_base,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,completion,left_right,A balanced explanation of the policy debate ar...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,it is necessary to protect the public from the...,42,0.0,False,128
4,economy_corporate_tax_neutral_arguments_base,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,completion,left_right,The debate about raising corporate tax rates h...,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,the tax rate is a tax on the productive capaci...,42,0.0,False,128


In [16]:
sample_prompt_id = raw_outputs["prompt_id"].iloc[0]
comparison = raw_outputs.loc[
    raw_outputs["prompt_id"].eq(sample_prompt_id),
    ["prompt_id", "model_type", "input_format", "prompt_text", "response_text"],
]

display(comparison)

,prompt_id,model_type,input_format,prompt_text,response_text
0,economy_wealth_tax_neutral_arguments,base,completion,The debate about increasing taxes on wealthy i...,the wealthy are not contributing enough to the...
200,economy_wealth_tax_neutral_arguments,instruct,instruction,What are the main arguments for and against in...,Increasing taxes on wealthy individuals is a c...
